# QiFeng-CYGNSS Dataset Visualization

This notebook demonstrates how to load and plot wind speed fields from the QiFeng-CYGNSS dataset.

**Grid**: 256 x 256 pixels, 1.5 km resolution, 384 km x 384 km storm-relative domain.

In [ ]:
import numpy as np
import netCDF4 as nc4
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Set path to your local copy of the dataset
DATASET_DIR = Path('/path/to/QiFeng_CYGNSS_dataset')

# Load Hurricane Ian (2022) as an example
ds = nc4.Dataset(str(DATASET_DIR / 'IAN_2022.nc'), 'r')

u10 = ds.variables['u10'][:]       # (time, 256, 256) eastward wind m/s
v10 = ds.variables['v10'][:]       # (time, 256, 256) northward wind m/s
center_lat = ds.variables['center_lat'][:]
center_lon = ds.variables['center_lon'][:]
ibt_vmax = ds.variables['ibt_vmax'][:]

ws = np.sqrt(u10**2 + v10**2)
print(f"Storm: {ds.storm_name}, Snapshots: {u10.shape[0]}, Max WS: {np.nanmax(ws):.1f} m/s")

In [ ]:
# Plot wind speed at peak intensity
t_idx = int(np.argmax(ibt_vmax))

# Convert pixel grid to geographic coordinates
clat, clon = float(center_lat[t_idx]), float(center_lon[t_idx])
half_km = 256 * 1.5 / 2.0
lat_arr = clat + np.linspace(half_km, -half_km, 256) / 110.574
lon_arr = clon + np.linspace(-half_km, half_km, 256) / (111.32 * np.cos(np.radians(clat)))
LON, LAT = np.meshgrid(lon_arr, lat_arr)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.pcolormesh(LON, LAT, ws[t_idx], cmap='RdBu_r', vmin=0, vmax=50, shading='auto')
ax.plot(clon, clat, 'k+', markersize=12, markeredgewidth=2)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'{ds.storm_name} ({ds.year}) | Vmax={ibt_vmax[t_idx]:.0f} kt')
ax.set_aspect('equal')
plt.colorbar(im, ax=ax, label='Wind Speed (m/s)', shrink=0.8)
plt.tight_layout()
plt.show()

In [ ]:
ds.close()